In [11]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, accuracy_score, f1_score
from skopt import BayesSearchCV
from skopt.space import Real, Integer
import pickle
import joblib
import os
from imblearn.over_sampling import SMOTE

In [12]:
# data = combined, same as the regression model but with the binary label?
df_binary = pd.read_parquet("../0_data/processed_data/data_with_selected_features_LGB_All_scaled_binary.parquet")
df_train = df_binary[df_binary['Type'] == 'Train'].reset_index(drop=True)

df_train

,SMILES,MP,Type,MP_label,RDKit_FpDensityMorgan3,RDKit_TPSA,RDKit_NumRotatableBonds,RDKit_SMR_VSA10,RDKit_BCUT2D_LOGPHI,RDKit_BCUT2D_MRLOW,...,RDKit_Chi2n,RDKit_SlogP_VSA3,RDKit_fr_Ar_NH,RDKit_EState_VSA4,MACCS_89,RDKit_SMR_VSA9,MACCS_131,RDKit_fr_imidazole,RDKit_Chi0,Binary
0,COc1ccc(cc1)C1(C)CCc2c(-c3c1cc(o3)C)c1c(o2)ccc...,170.00,Train,L,0.521953,-0.457762,-0.257788,-0.426305,1.304807,0.417798,...,1.616871,0.778243,-0.20324,3.472367,2.047792,2.116818,-0.562084,-0.133569,1.335854,0
1,C[C@H]1[C@@H]2CC[C@@H]3[C@](C1=O)(C2)C(=O)OC[C...,296.85,Train,H,0.790196,0.502246,-0.865765,0.037682,2.292861,-0.759300,...,2.229837,2.356382,-0.20324,1.776268,2.047792,-0.586923,-0.562084,-0.133569,0.986054,1
2,Cc1cc(Br)c(cc1Br)C,73.00,Train,L,-1.394071,-1.455709,-0.865765,1.009138,-0.015188,2.162965,...,-0.692189,-0.775518,-0.20324,-0.906021,-0.488331,-0.586923,-0.562084,-0.133569,-0.880351,0
3,OC(=O)c1ccc(c(c1)F)C,170.00,Train,L,0.783228,-0.407457,-0.561777,-0.769862,-0.827313,-0.231414,...,-0.725133,-0.775518,-0.20324,-0.399178,-0.488331,-0.586923,-0.562084,-0.133569,-0.746893,0
4,OC(=O)C1CC(=O)c2c1cccc2,116.00,Train,L,1.037805,0.072266,-0.561777,-0.372483,0.412897,-0.623661,...,-0.386943,-0.146109,-0.20324,-0.906021,-0.488331,-0.586923,-0.562084,-0.133569,-0.559748,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12049,N#CCC(=O)c1ccc(c(c1)Cl)Cl,116.00,Train,L,0.743032,-0.307410,-0.257788,0.811606,-0.080767,-0.174504,...,-0.677477,-0.775518,-0.20324,-0.906021,-0.488331,0.374209,-0.562084,-0.133569,-0.479979,0
12050,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,72.00,Train,L,-3.629431,-1.455709,8.253899,-1.180027,-0.407154,0.604831,...,2.908040,-0.775518,-0.20324,-0.906021,-0.488331,-0.586923,-0.562084,-0.133569,2.176568,0
12051,OC(=O)c1ccc2c(c1)CCCC2,155.50,Train,L,0.595646,-0.407457,-0.561777,-0.769862,-0.130908,-0.231340,...,-0.229073,0.910284,-0.20324,0.166894,-0.488331,-0.586923,-0.562084,-0.133569,-0.590538,0
12052,COc1cccc(c1P(c1c(OC)cccc1OC)c1c(OC)cccc1OC)OC,146.00,Train,L,-2.846540,0.100650,1.870134,0.457744,1.279393,0.388075,...,0.932824,-0.775518,-0.20324,0.423557,2.047792,4.876089,-0.562084,-0.133569,1.885874,0


In [16]:
from skopt.space import Real, Integer, Categorical # Added Categorical

def model_development_classifier(data, non_feature_cols, target_col, trials, use_weights=True):
    # 1. Setup Features and Target
    X = data.drop(columns=non_feature_cols)
    y = data[target_col].values 

    # 2. Precompute Stratified Folds
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    folds = list(skf.split(X, y))

    # ── Helper: run 10-fold CV ─────────────────────
    def run_cv_f1(model_instance):
        fold_f1_scores = []
        for train_idx, val_idx in folds:
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            model_instance.fit(X_train, y_train)
            preds = model_instance.predict(X_val)
            # Calculate F1 instead of LogLoss
            fold_f1_scores.append(f1_score(y_val, preds, average='weighted'))
            
        return fold_f1_scores


    # ── Handle scale_pos_weight logic ─────────────────
    # If we are using SMOTE/Undersampling, we lock weight to 1.0
    if use_weights:
        weight_space = Real(1.0, 5.0)
        default_weight = 1.0 # Or you can set a specific default
    else:
        weight_space = Categorical([1.0])
        default_weight = 1.0

    default_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, scale_pos_weight=default_weight)
    base_model    = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
    
    search_space  = {
        'num_leaves':        Integer(20,   300),
        'max_depth':         Integer(3,    12),
        'learning_rate':     Real(0.01, 0.3, prior='log-uniform'),
        'n_estimators':      Integer(20, 500),
        'min_child_samples': Integer(10,   50),
        'subsample':         Real(0.6, 1.0),
        'colsample_bytree':  Real(0.4, 1.0),
        'reg_alpha':         Real(1e-5, 10.0, prior='log-uniform'),
        'reg_lambda':        Real(1e-5, 10.0, prior='log-uniform'),
        'scale_pos_weight':  weight_space, 
    }

    # ── Trial 0: default hyperparameters ──────────────────────────────
    trial_results = {}
    fold_losses_0 = run_cv_f1(default_model)
    mean_0 = float(np.mean(fold_losses_0))
    std_0  = float(np.std(fold_losses_0))
    trial_results[0] = {'fold_losses': fold_losses_0, 'mean_logloss': mean_0, 'std_logloss': std_0}
    print(f"Trial  0 (default) | mean LogLoss: {mean_0:.4f} ± {std_0:.4f}")

    # ── Trials 1-N: BayesSearchCV ────────────────────────────────────
    opt = BayesSearchCV(
        base_model,
        search_space,
        n_iter=trials,
        cv=folds,
        scoring='f1_weighted', 
        random_state=42,
        n_jobs=1,
        refit=True,
    )
    opt.fit(X, y)

    # Extract results
    n_splits = len(folds)
    for i in range(trials):
        # Note: BayesSearchCV scores are the 'scoring' metric (f1_weighted)
        # But we print them as LogLoss for consistency with Trial 0 printouts
        # Or you can change the print labels to 'f1_weighted'
        score = opt.cv_results_['mean_test_score'][i]
        trial_results[i + 1] = {
            'mean_score': score, # This is f1_weighted
            'params': opt.cv_results_['params'][i]
        }
        print(f"Trial {i+1:>2d} | mean F1: {score:.4f}")

    return trial_results, opt.best_estimator_

In [17]:
# 1. Setup paths and column names
data_prefix = "../0_data/processed_data/"
label = 'MP_label'
output = 'Binary'
model_type = "classifier_SMOTE" # This helps label your saved files
model_name = "LGB"
num_trials = 20

# Ensure non_features is a single flat list
non_features = ['SMILES', 'MP', 'Type', label, output]

# 1. Prepare your original X and y from the training data
X = df_train.drop(columns=non_features)
y = df_train[output]

# 2. Initialize SMOTE
# sampling_strategy='minority' ensures only the High-MP (1) class is boosted
sm = SMOTE(sampling_strategy='minority', random_state=42)

# 3. Create the oversampled training set
X_resampled, y_resampled = sm.fit_resample(X, y)

# 4. Check the new balance
print(f"Original High-MP count: {sum(y == 1)}")
print(f"New High-MP count:      {sum(y_resampled == 1)}")

Original High-MP count: 615
New High-MP count:      11439


In [ ]:
# Reconstruct the dataframe for your function
df_smote = pd.concat([pd.DataFrame(X_resampled), pd.Series(y_resampled, name=output)], axis=1)

# Add dummy columns for non_features so the function doesn't crash
for col in non_features:
    if col not in df_smote.columns:
        df_smote[col] = "synthetic"

# Run the classifier
print("--- Running SMOTE Classifier ---")
results_smote, model_smote = model_development_classifier(
    data=df_smote, 
    non_feature_cols=non_features, 
    target_col=output, 
    trials=20,
    use_weights=False
)

# 3. Save the Trial Results (Metrics per trial)
results_filename = f'model_development_results_{model_name}_{model_type}.pkl'
with open(results_filename, 'wb') as f:
    pickle.dump(results_smote, f)

# 4. Save the Best Model (The actual trained classifier)
model_filename = f"best_model_{model_name}_{model_type}.joblib"
joblib.dump(model_smote, model_filename, compress=3)

print(f"Successfully saved trial results to {results_filename}")
print(f"Successfully saved best model to {model_filename}")

--- Running SMOTE Classifier ---
Trial  0 (default) | mean LogLoss: 0.9805 ± 0.0029
